# inkcheck in 5 minutes: from an official open-data ink map to a verdict

This notebook takes **official, public Vesuvius Challenge ink predictions** and turns them into a verdict, end to end, on a laptop CPU.

| Step | What | Expected |
|---|---|---|
| 1 | Open an official prediction map straight from the open-data bucket (`inkcheck.opendata`) | map + metadata (scan, model, pixel size) |
| 2 | **PASS-like:** the team's 2.2 um map of PHerc0343P vs letters traced on an *infrared photograph* (a reference that does not come from any model or from the CT) | `PASS`, and the wrong-place control `FAIL`s |
| 3 | Two official models on the same PHerc0172 segment, checked against each other | significant but only `WEAK` agreement; wrong-place control `FAIL`s |
| 4 | **WARN-like:** saturation risk of the PHerc0172 53 keV scan vs the 116 keV PHerc0343P scan | `HIGH` vs `LOW` |

**Data.** Everything is fetched at run time (about 15 MB, cached in `$INKCHECK_CACHE` or `~/.cache/inkcheck`). Scroll data are
CC BY-NC 4.0 with additional terms from the Vesuvius Challenge: **do not commit or redistribute** the downloaded files or images
drawn from them. This notebook ships without outputs and draws no images.

**Install:** `pip install git+https://github.com/dlescanogithub/vesuvius-inkcheck` (Python >= 3.10). Runtime: ~3-5 min on 4 CPU cores.

In [ ]:
import json, time
import numpy as np
from inkcheck import opendata as od
from inkcheck.core import CheckConfig, run_check
from inkcheck.quick import wrong_place
from inkcheck.node_d_v2 import saturation_check

WORKERS = 4          # CPU processes for the null tests
T0 = time.time()

## 1. Open an official prediction map

Official ink predictions live under `<scroll>/segments/<segment>/ink-detection/` (full-resolution `.tif`) and
`.../ink-detection/downsampled/` (`-ds8.jpg` previews, 8x smaller). The file name encodes the scan, the model and the tiling;
`parse_prediction_name` reads it, so the map's pixel size is known without guessing. `load_prediction` accepts a local path,
an `https://` URL, an `s3://` URL or a bare bucket key.

In [ ]:
SCROLL, SEGDIR = "PHerc0172", "20250917143559-w062_20250917143559205_flatboi"
keys = od.list_predictions(SCROLL, SEGDIR)            # anonymous S3 listing of the ds8 previews
for k in keys:
    print(k.rsplit("/", 1)[-1])

pick = lambda model: next(k for k in keys if model in k)
A_KEY, B_KEY = pick("july_retreat"), pick("november19")
a, meta_a = od.load_prediction(A_KEY)
b, meta_b = od.load_prediction(B_KEY)
print({k: meta_a[k] for k in ("scroll", "segment", "scan", "model", "ds", "map_pixel_um", "shape")})
print({k: meta_b[k] for k in ("model", "run", "shape")})

## 2. PASS-like: does a model map see *known* ink?

A verdict needs a reference that the model could not have produced itself. The built-in `343p-ir` example uses the
letters the Vesuvius Challenge team traced on an **infrared photograph** of fragment PHerc0343P (carbon ink is black in IR),
placed on the 8.64 um scan by D. Russo (public repo, pinned commit, sha256-checked). The prediction is the team's canonical
2.2 um model map of the same segment, resampled onto that canvas.

`inkcheck quick` runs node A (band-pass correlation and stroke recall against 192 shifted/rotated copies of the prediction,
i.e. "how often would a map this textured agree by chance?") and node N: the **same** prediction moved to the wrong place must FAIL.

In [ ]:
from inkcheck.quick import run as quick
rc = quick(["343p-ir", "--workers", str(WORKERS), "--out", "out/tutorial/343p-ir"])
s = json.load(open("out/tutorial/343p-ir.summary.json"))
print("\nverdict:", s["verdict"], "| node A:", s["node_A"], "| control:", s["control_verdict"])
assert s["verdict"] == "PASS" and s["control_verdict"] == "FAIL", s

## 3. Two official models on the same PHerc0172 segment

Here there is no ground truth, so the most we can ask is whether two **independent** official models
(`july_retreat`, 2025-07, and `november19`, 2025-12) put ink in the same places. Both maps are on the same
63.28 um grid (7.91 um x ds8), so no registration is needed. Expect a clearly significant band-pass correlation but only
**WEAK** at letter level: at ds8 the strokes are a few pixels wide, and agreement between two models is not evidence that
either sees real ink. The wrong-place control must still FAIL.

In [ ]:
um = meta_a["map_pixel_um"]
cfg = CheckConfig(pred_um=um, ref_um=um, workers=WORKERS)
valid = a > 0                                        # outside the segment the preview is black
res = run_check(a, b, cfg, valid, None, log=lambda *_: None)
ctrl = run_check(wrong_place(a), b, cfg, wrong_place(valid), None, log=lambda *_: None)
n = res.numbers
print(f"node A: {res.verdict}  r_bp={n['r_bandpass']:.3f} (p={n['r_bandpass_p']:.3g}, null mean {n['r_bandpass_null_mean']:.3f}); "
      f"stroke recall {n['stroke_recall']:.0%}")
print(f"wrong-place control: {ctrl.verdict}  r_bp={ctrl.numbers['r_bandpass']:.3f}")
assert ctrl.verdict == "FAIL"

## 4. WARN-like: is the scan itself a risk?

PHerc0172 was scanned at 53 keV; most training data are 113-116 keV. Its 8-bit surface volume is much brighter and flatter,
and in our F1 study Reader V3 saturated on it (56-67% of the map above the high level vs 0-26% elsewhere), which makes
top-quantile agreement tests unreliable. `open_surface_volume` reads a 512 x 512 window of the centre layer of each
official surface volume (a few 128 x 128 chunks, ~1 MB) and `ct_saturation_level` applies the same fixed thresholds as
`scripts/saturation_risk_table.py`. `saturation_check` then looks at the prediction map itself.

In [ ]:
SV = {
    "PHerc0172 53 keV": f"{SCROLL}/segments/{SEGDIR}/surface-volumes/7.91um-53keV-volume-20241024131838.zarr/",
    "PHerc0343P 116 keV": "PHerc0343P/segments/20250511003658-tifxyz/surface-volumes/8.64um-1.2m-116keV-volume-20250521134555.zarr",
}
levels = {}
for name, key in SV.items():
    sv = od.open_surface_volume(key)
    H, W = sv.shape[1:]
    blk = sv.read(sv.centred_window(1), slice(H // 2 - 256, H // 2 + 256), slice(W // 2 - 256, W // 2 + 256))
    st = od.ct_intensity_stats(blk)
    levels[name] = od.ct_saturation_level(st)
    print(f"{name:20s} shape {sv.shape}  CT median {st['median']:.0f}, share>128 {st['frac_gt128']:.2f} -> risk {levels[name]}")

sat = saturation_check(a, valid)
print(f"official july_retreat map: share above high level {sat['frac_high']:.2f} -> saturated={sat['saturated']}")
verdict4 = "WARN" if levels["PHerc0172 53 keV"] == "HIGH" else "OK"
print("PHerc0172 scan:", verdict4, "- check any model's map on this scan with saturation_check before trusting a PASS")

## Summary

In [ ]:
print(f"1-2  PHerc0343P team map vs infrared letters : {s['verdict']} (control {s['control_verdict']})")
print(f"3    PHerc0172 july_retreat vs november19     : {res.verdict} (control {ctrl.verdict})")
print(f"4    PHerc0172 53 keV scan saturation risk     : {levels['PHerc0172 53 keV']} -> {verdict4}; "
      f"PHerc0343P 116 keV: {levels['PHerc0343P 116 keV']}")
print(f"total {time.time() - T0:.0f} s")

**Next:** your own map: `inkcheck quick PRED REF --pred-um .. --ref-um ..`; a full report: `inkcheck PRED REF ...`;
API reference for the adapter: `help(inkcheck.opendata)`; see `docs/TUTORIAL.md`.